# Train a Model for Detecting Solar Panels

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/train_solar_panel_detection.ipynb)

## Install package
To use the `geoai-py` package, ensure it is installed in your environment. Uncomment the command below if needed.

In [ ]:
# %pip install geoai-py

## Import libraries

In [1]:
import geoai

## Download sample data

In [2]:
train_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/solar_panels_davis_ca.tif"
train_vector_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/solar_panels_davis_ca.geojson"
test_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/solar_panels_test_davis_ca.tif"

In [3]:
train_raster_path = geoai.download_file(train_raster_url)
train_vector_path = geoai.download_file(train_vector_url)
test_raster_path = geoai.download_file(test_raster_url)

## Visualize sample data

In [4]:
geoai.view_vector_interactive(train_vector_path, tiles=train_raster_url)

In [5]:
geoai.view_raster(test_raster_url)

## Create training data

In [6]:
out_folder = "output"
tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_vector_path,
    tile_size=512,
    stride=256,
    buffer_radius=0,
)


Raster info for solar_panels_davis_ca.tif:
  CRS: EPSG:3857
  Dimensions: 6711 x 3242
  Resolution: (0.07464427187774597, 0.07464193859333511)
  Bands: 3
  Bounds: BoundingBox(left=-13556075.706688922, bottom=4657729.5446374295, right=-13555574.76898035, top=4657971.533802349)
Loaded 67 features from solar_panels_davis_ca.geojson
Vector CRS: EPSG:4326
Reprojecting features from EPSG:4326 to EPSG:3857


Generated: 312, With features: 137: 100%|██████████| 312/312 [00:16<00:00, 19.22it/s]


------- Export Summary -------
Total tiles exported: 312
Tiles with features: 137 (43.9%)
Average feature pixels per tile: 8422.7
Output saved to: output

------- Georeference Verification -------


## Train object detection model

In [8]:
geoai.train_MaskRCNN_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/models",
    num_channels=3,
    pretrained=True,
    batch_size=4,
    num_epochs=100,
    learning_rate=0.005,
    val_split=0.2,
)

Using device: cuda
Found 312 image files and 312 label files
Training on 249 images, validating on 63 images
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to C:\Users\tusha/.cache\torch\hub\checkpoints\maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:58<00:00, 3.05MB/s] 


Epoch: 1, Batch: 1/63, Loss: 2.5753, Time: 1.18s
Epoch: 1, Batch: 11/63, Loss: 0.4112, Time: 5.03s
Epoch: 1, Batch: 21/63, Loss: 0.6464, Time: 4.67s
Epoch: 1, Batch: 31/63, Loss: 0.1568, Time: 4.90s
Epoch: 1, Batch: 41/63, Loss: 0.3155, Time: 4.76s
Epoch: 1, Batch: 51/63, Loss: 0.3366, Time: 4.67s
Epoch: 1, Batch: 61/63, Loss: 0.3848, Time: 7.06s
Epoch 1/100: Train Loss: 0.5036, Val Loss: inf, Val IoU: 0.3011
Saving best model with IoU: 0.3011
Epoch: 2, Batch: 1/63, Loss: 0.3522, Time: 0.37s
Epoch: 2, Batch: 11/63, Loss: 0.2363, Time: 3.90s
Epoch: 2, Batch: 21/63, Loss: 0.1609, Time: 3.81s
Epoch: 2, Batch: 31/63, Loss: 0.1607, Time: 3.81s
Epoch: 2, Batch: 41/63, Loss: 0.1331, Time: 3.75s
Epoch: 2, Batch: 51/63, Loss: 0.1800, Time: 3.74s
Epoch: 2, Batch: 61/63, Loss: 0.1979, Time: 3.93s
Epoch 2/100: Train Loss: 0.2164, Val Loss: inf, Val IoU: 0.2207
Epoch: 3, Batch: 1/63, Loss: 0.1402, Time: 0.37s
Epoch: 3, Batch: 11/63, Loss: 0.2023, Time: 3.86s
Epoch: 3, Batch: 21/63, Loss: 0.3315, Ti

## Run inference

In [9]:
masks_path = "solar_panels_prediction.tif"
model_path = f"{out_folder}/models/best_model.pth"

In [10]:
geoai.object_detection(
    test_raster_path,
    masks_path,
    model_path,
    window_size=512,
    overlap=256,
    confidence_threshold=0.5,
    batch_size=4,
    num_channels=3,
)

Processing 209 windows with size 512x512 and overlap 256...


240it [00:10, 23.26it/s]                         


Inference completed in 10.36 seconds
Saved prediction to solar_panels_prediction.tif


## Vectorize masks

In [11]:
output_path = "solar_panels_prediction.geojson"
gdf = geoai.orthogonalize(masks_path, output_path, epsilon=2)

Processing 42 features...


Converting features: 100%|██████████| 42/42 [00:00<00:00, 200.96shape/s]

Saving to solar_panels_prediction.geojson...
Done!


## Visualize results

In [12]:
geoai.view_vector_interactive(output_path, tiles=test_raster_url)

In [13]:
geoai.create_split_map(
    left_layer=output_path,
    right_layer=test_raster_url,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_raster_url,
)

Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
Expecting value: line 1 column 1 (char 0)
The provided layers are invalid!


ValueError: The 'url' trait of a TileLayer instance expected a unicode string, not the NoneType None.

![image](https://github.com/user-attachments/assets/69d20aec-991f-4681-b31e-fabcd7e21b91)